<a href="https://colab.research.google.com/github/seeuny2022-commits/finflag/blob/main/Seeun_PARK_FinFlag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install google-genai datasets gradio yfinance

In [3]:
import json, time
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata, files

# Read the secret key (never paste the key itself here, because this code goes on GitHub)
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# Find the "flash" models your key can use (fast + free)
names = [m.name.replace("models/", "") for m in client.models.list()]
flash = [n for n in names if "flash" in n and not any(x in n for x in ["image", "tts", "live", "audio", "embed", "lite"])]
print("Flash models available:", flash)

MODEL = flash[0]
print("Using:", MODEL)

In [ ]:
from datasets import load_dataset

ds = load_dataset("gtfintechlab/VideoConviction")
print(ds)

df = ds[list(ds.keys())[0]].to_pandas()   # turn it into a table
print(df.shape)                            # (rows, columns)
print(df.columns.tolist())                 # the column names
df.head(3)

In [ ]:
TEXT_COL = "segment_transcript" if "segment_transcript" in df.columns else "transcript"
VIDEO_COL = "youtube_video_url"
CHANNEL_COL = next((c for c in df.columns if "channel" in c.lower()), None)
print("text:", TEXT_COL, "| video:", VIDEO_COL, "| channel:", CHANNEL_COL)

# Remove empty or very short segments
df = df[df[TEXT_COL].fillna("").str.len() > 80].reset_index(drop=True)
df["seg_id"] = range(len(df))
print(len(df), "usable segments from", df[VIDEO_COL].nunique(), "videos")

print("\nExample segment:\n", df.loc[0, TEXT_COL][:800])

In [ ]:
FLAGS = ["GUARANTEE", "URGENCY", "NO_RISK", "CERTAINTY", "PROMOTION", "ALL_IN"]

# Shuffle, then keep only ONE segment per video (so the same video can't be in both dev and test)
one_per_video = df.sample(frac=1, random_state=42).drop_duplicates(VIDEO_COL)

# Pick 70: first 20 = dev (for improving the prompt), last 50 = test (locked until the end)
sample = one_per_video.sample(min(70, len(one_per_video)), random_state=42).reset_index(drop=True)
sample["split"] = ["dev"] * 20 + ["test"] * (len(sample) - 20)

sheet = sample[["seg_id", "split", VIDEO_COL, TEXT_COL]].copy()
sheet = sheet.rename(columns={TEXT_COL: "text", VIDEO_COL: "video"})
for f in FLAGS + ["RISKY"]:
    sheet[f] = ""          # empty columns for YOU to fill in

sheet.to_csv("labeling_sheet.csv", index=False)
files.download("labeling_sheet.csv")
print(sheet.split.value_counts())